# 中国银行 (601988.SH) 量化分析 Notebook

**流程：** Tushare 获取数据 → 数据清洗 → 计算技术指标 → 画K线图 → 基本面+技术面分析

**作者：** BA-Quant &nbsp;|&nbsp; **日期：** 2026-07-01

## 1. 环境准备

安装并导入所需依赖：

In [ ]:
# 安装依赖（首次运行取消注释）
# !pip install tushare pandas numpy plotly matplotlib

In [ ]:
import tushare as ts
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

print("✅ 环境准备完成")

## 2. 从 Tushare 获取中国银行近一年日线数据

股票代码：**601988.SH**（中国银行 A 股）
时间范围：**2025-07-01 ~ 2026-07-01**

> 注意：需先注册 [Tushare Pro](https://tushare.pro) 获取 Token

In [ ]:
# ===== 设置 Tushare Token =====
# ts.set_token('YOUR_TOKEN_HERE')   # 替换为你的 Token
# pro = ts.pro_api()

# ===== 方法1：Tushare Pro API（推荐）=====
TS_CODE = '601988.SH'
START_DATE = '20250701'
END_DATE = '20260701'

# 取消注释以实际获取：
# df = pro.daily(ts_code=TS_CODE, start_date=START_DATE, end_date=END_DATE)
# df = df.rename(columns={'trade_date': 'date', 'vol': 'volume'})
# df = df.sort_values('date').reset_index(drop=True)

# ===== 方法2：基础接口 =====
# df = ts.get_hist_data('601988', start='2025-07-01', end='2026-07-01')
# df = df.sort_index().reset_index()
# df.columns = ['date', 'open', 'high', 'close', 'low', 'volume', 'price_change',
#               'pct_change', 'ma5', 'ma10', 'ma20', 'v_ma5', 'v_ma10', 'v_ma20']

print("数据获取完成！")
print(f"字段：{list(df.columns)}")
print(f"数据量：{len(df)} 条")

## 3. 读取本地 CSV 数据（离线模式）

如果已经将数据保存为 CSV，也可以直接读取本地文件：

In [ ]:
# 读取本地 CSV
csv_path = '601988_daily.csv'
df = pd.read_csv(csv_path, parse_dates=['date'])

print(f"从 CSV 读取：{len(df)} 条数据")
print(df.head(3))

# 数据概览
print("\n--- 数据统计 ---")
print(df.describe())

## 4. 计算技术指标

计算 MA（移动平均线）、BOLL（布林带）、RSI、MACD 等常用指标：

In [ ]:
# ===== 移动平均线 =====
def calc_ma(series, period):
    return series.rolling(window=period).mean()

for p in [5, 10, 20, 60, 120]:
    df[f'ma{p}'] = calc_ma(df['close'], p)

# ===== 布林带 (20, 2) =====
df['boll_mid'] = df['close'].rolling(20).mean()
std = df['close'].rolling(20).std()
df['boll_upper'] = df['boll_mid'] + 2 * std
df['boll_lower'] = df['boll_mid'] - 2 * std

# ===== RSI(14) =====
def calc_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df['rsi'] = calc_rsi(df['close'], 14)

# ===== MACD =====
ema12 = df['close'].ewm(span=12, adjust=False).mean()
ema26 = df['close'].ewm(span=26, adjust=False).mean()
df['dif'] = ema12 - ema26
df['dea'] = df['dif'].ewm(span=9, adjust=False).mean()
df['macd_bar'] = 2 * (df['dif'] - df['dea'])

print("技术指标计算完成 ✅")
df[['date','close','ma5','ma10','ma20','boll_upper','boll_lower','rsi','dif','dea']].tail(10)


## 5. 绘制 K 线图（使用 Plotly）

### 5.1 K 线图 + 均线 + 布林带

In [ ]:
# 创建 K 线图
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.55, 0.2, 0.25],
    subplot_titles=('K线图 + 均线 + 布林带', '成交量', 'MACD')
)

# ---- 子图1：K线 + 均线 + 布林带 ----
fig.add_trace(go.Candlestick(
    x=df['date'],
    open=df['open'], high=df['high'],
    low=df['low'], close=df['close'],
    name='K线',
    increasing_line_color='#ef4444',
    decreasing_line_color='#22c55e'
), row=1, col=1)

# 均线
colors = {'ma5': '#f59e0b', 'ma10': '#06b6d4', 'ma20': '#a855f7', 'ma60': '#ec4899'}
for name, color in colors.items():
    fig.add_trace(go.Scatter(
        x=df['date'], y=df[name],
        mode='lines', name=name.upper(),
        line=dict(width=1, color=color)
    ), row=1, col=1)

# 布林带
for band, style in [('boll_upper', 'dash'), ('boll_lower', 'dash'), ('boll_mid', 'dot')]:
    fig.add_trace(go.Scatter(
        x=df['date'], y=df[band],
        mode='lines', name=f'BOLL',
        line=dict(width=0.8, color='rgba(59,130,246,0.5)', dash=style),
        showlegend=False
    ), row=1, col=1)

# ---- 子图2：成交量 ----
colors_vol = ['#ef4444' if df['close'].iloc[i] >= df['open'].iloc[i] else '#22c55e'
              for i in range(len(df))]
fig.add_trace(go.Bar(
    x=df['date'], y=df['volume'],
    name='成交量',
    marker_color=colors_vol,
    showlegend=False
), row=2, col=1)

# ---- 子图3：MACD ----
fig.add_trace(go.Scatter(
    x=df['date'], y=df['dif'],
    mode='lines', name='DIF',
    line=dict(width=1, color='#f59e0b')
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=df['date'], y=df['dea'],
    mode='lines', name='DEA',
    line=dict(width=1, color='#06b6d4')
), row=3, col=1)

macd_colors = ['#ef4444' if v >= 0 else '#22c55e' for v in df['macd_bar']]
fig.add_trace(go.Bar(
    x=df['date'], y=df['macd_bar'],
    name='MACD',
    marker_color=macd_colors,
    showlegend=False
), row=3, col=1)

# 布局
fig.update_layout(
    title='中国银行 (601988.SH) 近一年 K 线图',
    template='plotly_dark',
    height=900,
    xaxis_rangeslider_visible=False,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='top', y=1.12, xanchor='left', x=0),
    margin=dict(t=60, b=20, l=60, r=40)
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='rgba(128,128,128,0.15)')

fig.show()
print("K线图绘制完成 ✅")

### 5.2 RSI 指标图

In [ ]:
fig_rsi = go.Figure()

fig_rsi.add_trace(go.Scatter(
    x=df['date'], y=df['rsi'],
    mode='lines', name='RSI(14)',
    line=dict(width=1.5, color='#a855f7')
))

# 超买超卖线
fig_rsi.add_hline(y=70, line_dash='dash', line_color='rgba(239,68,68,0.5)',
                  annotation_text='超买 (70)', annotation_position='top right')
fig_rsi.add_hline(y=30, line_dash='dash', line_color='rgba(34,197,94,0.5)',
                  annotation_text='超卖 (30)', annotation_position='bottom right')
fig_rsi.add_hline(y=50, line_dash='dot', line_color='rgba(128,128,128,0.3)')

fig_rsi.update_layout(
    title='RSI(14) 指标',
    template='plotly_dark',
    height=350,
    margin=dict(t=40, b=20, l=60, r=40),
    yaxis=dict(range=[0, 100])
)
fig_rsi.show()

## 6. 保存数据到本地

In [ ]:
# 保存为 CSV
df.to_csv('601988_daily.csv', index=False, encoding='utf-8-sig')
print("数据已保存为 601988_daily.csv ✅")

# 保存为 JSON
df.to_json('601988_daily.json', orient='records', force_ascii=False, indent=2)
print("数据已保存为 601988_daily.json ✅")

## 7. 基本面数据分析

In [ ]:
# 计算关键统计量
latest = df.iloc[-1]
prev = df.iloc[-2]
max_52w = df['high'].max()
min_52w = df['low'].min()
avg_vol_20 = df['volume'].tail(20).mean()

print("=" * 50)
print(" 中国银行 (601988.SH) 基础数据汇总")
print("=" * 50)
print(f" 最新收盘价：  ¥{latest['close']:.2f}")
print(f" 涨跌幅（日）：  {((latest['close'] - prev['close']) / prev['close'] * 100):+.2f}%")
print(f" 52周最高价：   ¥{max_52w:.2f}")
print(f" 52周最低价：   ¥{min_52w:.2f}")
print(f" 52周涨跌幅：   {((latest['close'] - df.iloc[0]['close']) / df.iloc[0]['close'] * 100):+.2f}%")
print(f" 近20日均量：   {avg_vol_20/10000:.0f} 万手")
print(f" RSI(14)：      {latest['rsi']:.1f}")
print(f" 当前 MA20：    ¥{latest['ma20']:.2f}")
print(f" 当前 MA60：    ¥{latest['ma60']:.2f}")

# 基本面数据（来源 2025年报 / 2026Q1）
print()
print("=" * 50)
print(" 基本面数据")
print("=" * 50)
fundamentals = {
    '2025年营收': '6,599亿 (+4.28% YoY)',
    '2025年归母净利': '2,430亿 (+2.18% YoY)',
    '2026Q1营收': '1,788亿 (+8.4% YoY)',
    '2026Q1归母净利': '566亿 (+4.2% YoY)',
    'ROE (2025)': '8.94%',
    '净息差 (2025)': '1.26%',
    '不良率': '1.23% (六大行最低)',
    '当前PB': '≈0.62x',
    '股息率': '≈4.0%',
}
for k, v in fundamentals.items():
    print(f" {k:　<12s}:  {v}")

## 8. 总结

本 Notebook 完成了以下流程：

| 步骤 | 内容 |
|------|------|
| 1. 环境准备 | 安装 tushare / pandas / plotly |
| 2. 数据获取 | Tushare Pro API 获取 601988.SH 近一年日线数据 |
| 3. 数据读取 | 从本地 CSV 读取离线数据 |
| 4. 技术指标 | 计算 MA5/10/20/60、BOLL、RSI、MACD |
| 5. K线图 | Plotly 交互式 K线图 + 成交量 + MACD 三面板 |
| 6. RSI图 | RSI(14) 超买超卖分析 |
| 7. 数据存储 | CSV + JSON 双格式导出 |
| 8. 基本面分析 | 营收/净利/ROE/息差/不良率/PB/股息汇总 |

**扩展建议：**
- 添加更多股票对比分析
- 接入东方财富/新浪财经实时行情
- 添加量化策略回测（均线交叉、布林带突破等）
- 生成每日自动分析邮件报告